# 16. Train pooled calibration predictor

This notebook is the first calibration-training experiment with the universal pooled architecture:

\[
\text{all measurement tokens}
\rightarrow
\text{global Transformer}
\rightarrow
\text{shared calibration-wise attention pooler}
\rightarrow
\text{shared MLP}
\rightarrow
\text{shared correction head}.
\]

`calibration_ids` are routing metadata only. Their meaning is defined by the explicit ordered `CALIBRATION_KEYS` tuple and they are never embedded into the numerical token features. The learned parameter count is independent of the number of physical calibrations.

For this first run, numeric calibration context is not concatenated into the head, observability is absent, paired `+delta/-delta` training is absent, and normalized MSE is used. The augmentation distribution is kept aligned with the latest notebook 03 MSE run so the main experimental change is the architecture.

## 1. Imports and repository root

In [1]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "obscalib").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the nn-obs-calib repository root containing src/obscalib.")

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from kaist_dataset.dataset import KAISTCalibrationDataset, KAISTRigCalibration, build_kaist_sequence_specs, make_kaist_dataloader
from obscalib.augmentations.config import AugmentationConfig, CalibrationEventConfig, FrameRandomizationConfig, NoiseAugmentationConfig, PerturbationMagnitudeConfig, PriorPerturbationConfig, SamplingRateAugmentationConfig, SE3NoiseConfig, TrueCalibrationAugmentationConfig, VectorNoiseConfig
from obscalib.augmentations.pipeline import AugmentationPipeline
from obscalib.calibration.state import CalibrationState
from obscalib.calibration.update import CalibrationUpdater
from obscalib.config import CalibrationPoolerConfig, MLPConfig, PooledCalibrationHeadConfig, PooledCalibrationModelConfig, TransformerConfig, WindowingConfig
from obscalib.data.collation import collate_windows
from obscalib.data.device import window_batch_to
from obscalib.data.structures import GeometryType, MeasurementType, SensorStreamBatch, WindowBatch
from obscalib.geometry.lie import se3_log, so3_exp
from obscalib.geometry.processing import GeometryProcessor
from obscalib.models.pooled_calibration_model import PooledCalibrationModel
from obscalib.pipeline.window_step import WindowStep
from obscalib.tokenization.tokenizer import Tokenizer
from obscalib.training.checkpoint import load_training_checkpoint, save_training_checkpoint
from obscalib.training.metrics_logger import MetricsLogger

print("Repository root:", ROOT)
print("PyTorch:", torch.__version__)

Repository root: /home/camel/Skoltech/Mobile_Robotics_Lab/nn-obs-calib
PyTorch: 2.7.1+cu126


## 2. Experiment parameters

In [2]:
RUN_NAME = "kaist_pooled_calibration_mse_001"

RANDOM_SEED = 12345
TRAIN_AUGMENTATION_SEED = 23456
VALIDATION_AUGMENTATION_SEED = 34567
TEST_AUGMENTATION_SEED = 45678
VISUALIZATION_AUGMENTATION_SEED = 56789

DEVICE_NAME = "cuda"
DEVICE = torch.device("cuda" if DEVICE_NAME == "auto" and torch.cuda.is_available() else DEVICE_NAME)
MODEL_DTYPE = torch.float32

# Stable routing order for this rig. IDs are tuple positions only.
CALIBRATION_KEYS = ("imu", "lidar")

WINDOW_DURATION_S = 5.0
WINDOW_STRIDE_S = 5.0
MAX_SAMPLES_PER_SENSOR = 700

BATCH_SIZE = 16
NUM_WORKERS = 0
DROP_LAST_TRAIN = True

NUM_EPOCHS = 200
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
CHECKPOINT_EVERY_EPOCHS = 3
PRINT_EVERY_BATCHES = 500
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE_EPOCHS = 10

MEASUREMENT_DIM = 6
TOKEN_INPUT_DIM = MEASUREMENT_DIM + 2
D_MODEL = 128
N_HEADS = 8
NUM_TRANSFORMER_LAYERS = 12
DIM_FEEDFORWARD = 512
TRANSFORMER_DROPOUT = 0.1
NUM_SUMMARY_TOKENS = 1

POOLER_N_HEADS = 8
POOLER_DROPOUT = 0.1
MLP_HIDDEN_DIMS = (256, 128)
SHARED_FEATURE_DIM = 128
MLP_DROPOUT = 0.1
HEAD_HIDDEN_DIMS = (128, 64)
HEAD_DROPOUT = 0.1

CORRECTION_LOSS_TYPE = "mse"
ROTATION_LOSS_SCALE_RAD = np.deg2rad(5.0)
TRANSLATION_LOSS_SCALE_M = 0.05
TIME_OFFSET_LOSS_SCALE_S = 0.015
CHANGE_TIME_LOSS_SCALE_S = WINDOW_DURATION_S
ROTATION_LOSS_WEIGHT = 2.0
TRANSLATION_LOSS_WEIGHT = 1.0
TIME_OFFSET_LOSS_WEIGHT = 1.0
CHANGE_EVENT_LOSS_WEIGHT = 0.1
CHANGE_TIME_LOSS_WEIGHT = 0.1

RESUME_CHECKPOINT = None

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Device:", DEVICE)
print("Calibration order:", CALIBRATION_KEYS)
print("Correction loss:", CORRECTION_LOSS_TYPE)
print("Transformer layers:", NUM_TRANSFORMER_LAYERS)

Device: cuda
Calibration order: ('imu', 'lidar')
Correction loss: mse
Transformer layers: 12


## 3. KAIST split, calibration, datasets, and loaders

In [3]:
KAIST_DATASET_ROOT = Path("/mnt/d/Downloads/MobRobLab/KAISTDataset")
LIDAR_POSE_ROOT = Path("/home/camel/Skoltech/phd_proposal/data/KAISTDataset")

TRAIN_NAMES = ["Urban08", "Urban12", "Urban13", "Urban14", "Urban15", "Urban16"]
VAL_NAMES = ["Urban26", "Urban28"]
TEST_NAMES = ["Urban38", "Urban39"]
ALL_SEQUENCE_NAMES = TRAIN_NAMES + VAL_NAMES + TEST_NAMES

if len(ALL_SEQUENCE_NAMES) != len(set(ALL_SEQUENCE_NAMES)):
    raise ValueError("Train, validation, and test sequence sets overlap.")

T_B_L = np.array([
    [-0.514066, -0.702201, -0.492595, -0.440699],
    [0.486485, -0.711672, 0.506809, 0.397052],
    [-0.706447, 0.0208933, 0.707457, 1.90953],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

T_I_B = np.array([
    [1.0, 0.0, 0.0, -0.07],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.7],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

rig_calibration = KAISTRigCalibration(T_B_I=np.linalg.inv(T_I_B), T_B_L=T_B_L, tau_imu_s=0.0, tau_lidar_s=0.0)
windowing_config = WindowingConfig(window_duration_s=WINDOW_DURATION_S, window_stride_s=WINDOW_STRIDE_S, max_samples_per_sensor=MAX_SAMPLES_PER_SENSOR)

print("train:", TRAIN_NAMES)
print("validation:", VAL_NAMES)
print("test:", TEST_NAMES)

train: ['Urban08', 'Urban12', 'Urban13', 'Urban14', 'Urban15', 'Urban16']
validation: ['Urban26', 'Urban28']
test: ['Urban38', 'Urban39']


In [4]:
all_specs = build_kaist_sequence_specs(sequence_names=ALL_SEQUENCE_NAMES, kaist_dataset_root=KAIST_DATASET_ROOT, lidar_pose_root=LIDAR_POSE_ROOT)
spec_by_name = {spec.name: spec for spec in all_specs}

train_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TRAIN_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
val_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in VAL_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
test_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TEST_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)

print(train_dataset.summary())
print()
print(val_dataset.summary())
print()
print(test_dataset.summary())

KAISTCalibrationDataset: 2005 windows from 6 sequences
approximate tensor storage: 37.4 MiB
  Urban08: windows=59, duration=297.8s, IMU=30800, LiDAR=2952
  Urban12: windows=473, duration=2478.6s, IMU=249198, LiDAR=23222
  Urban13: windows=223, duration=1446.5s, IMU=145492, LiDAR=10536
  Urban14: windows=286, duration=1737.0s, IMU=174681, LiDAR=13638
  Urban15: windows=184, duration=1034.3s, IMU=104795, LiDAR=9011
  Urban16: windows=780, duration=4458.2s, IMU=449215, LiDAR=37703

KAISTCalibrationDataset: 417 windows from 2 sequences
approximate tensor storage: 7.8 MiB
  Urban26: windows=94, duration=567.0s, IMU=58371, LiDAR=4553
  Urban28: windows=323, duration=1958.6s, IMU=197456, LiDAR=15651

KAISTCalibrationDataset: 699 windows from 2 sequences
approximate tensor storage: 13.1 MiB
  Urban38: windows=349, duration=2147.3s, IMU=216225, LiDAR=17005
  Urban39: windows=350, duration=1856.3s, IMU=186682, LiDAR=17195


In [5]:
train_shuffle_generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = make_kaist_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=DROP_LAST_TRAIN, num_workers=NUM_WORKERS, generator=train_shuffle_generator)
val_loader = make_kaist_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)
test_loader = make_kaist_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)

print("train batches:", len(train_loader))
print("validation batches:", len(val_loader))
print("test batches:", len(test_loader))

train batches: 125
validation batches: 27
test batches: 44


## 4. Synthetic calibration problem

In [6]:
USE_SAMPLING_RATE_AUGMENTATION = False
USE_TRUE_CALIBRATION_AUGMENTATION = True
USE_CALIBRATION_EVENTS = False

augmentation_config = AugmentationConfig(
    sampling_rate=SamplingRateAugmentationConfig(enabled=USE_SAMPLING_RATE_AUGMENTATION, minimum_imu_frequency_hz=40.0, minimum_lidar_frequency_hz=5.0, imu_probability=0.8, lidar_probability=0.5),
    frame_randomization=FrameRandomizationConfig(enabled=False),
    true_calibration=TrueCalibrationAugmentationConfig(enabled=USE_TRUE_CALIBRATION_AUGMENTATION, probability=1.0, rotation_probability=1.0, time_offset_probability=0.0, rotation=PerturbationMagnitudeConfig(maximum=np.deg2rad(180.0)), time_offset=None),
    prior_perturbation=PriorPerturbationConfig(enabled=True, probability=1.0, rotation=PerturbationMagnitudeConfig(maximum=np.deg2rad(60.0)), translation=PerturbationMagnitudeConfig(maximum=0.15), time_offset=PerturbationMagnitudeConfig(maximum=0.15)),
    calibration_event=CalibrationEventConfig(enabled=USE_CALIBRATION_EVENTS),
    noise=NoiseAugmentationConfig(
        enabled=True,
        vector_by_type={
            MeasurementType.IMU_GYROSCOPE: VectorNoiseConfig(gaussian_std=0.01, window_bias_std=0.003),
            MeasurementType.IMU_ACCELEROMETER: VectorNoiseConfig(gaussian_std=0.08, window_bias_std=0.03),
        },
        se3_by_type={MeasurementType.LIDAR_POSE: SE3NoiseConfig(rotation_std=np.deg2rad(0.15), translation_std=0.01)},
    ),
)

augmentation_pipeline = AugmentationPipeline(augmentation_config)
print(augmentation_config)

AugmentationConfig(sampling_rate=SamplingRateAugmentationConfig(enabled=False, minimum_imu_frequency_hz=40.0, minimum_lidar_frequency_hz=5.0, imu_probability=0.8, lidar_probability=0.5), frame_randomization=FrameRandomizationConfig(enabled=False), prior_perturbation=PriorPerturbationConfig(enabled=True, probability=1.0, probability_by_key={}, rotation=PerturbationMagnitudeConfig(maximum=1.0471975511965976, distribution=<PerturbationDistribution.UNIFORM: 'uniform'>, scale=None, degrees_of_freedom=3), translation=PerturbationMagnitudeConfig(maximum=0.15, distribution=<PerturbationDistribution.UNIFORM: 'uniform'>, scale=None, degrees_of_freedom=3), time_offset=PerturbationMagnitudeConfig(maximum=0.15, distribution=<PerturbationDistribution.UNIFORM: 'uniform'>, scale=None, degrees_of_freedom=3)), calibration_event=CalibrationEventConfig(enabled=False, event_probability=0.5, event_probability_by_key={}, rotation_probability=0.0, translation_probability=0.0, time_offset_probability=0.0, rota

## 5. Pooled model and complete window pipeline

In [7]:
model_config = PooledCalibrationModelConfig(
    transformer=TransformerConfig(input_dim=TOKEN_INPUT_DIM, d_model=D_MODEL, n_heads=N_HEADS, num_layers=NUM_TRANSFORMER_LAYERS, dim_feedforward=DIM_FEEDFORWARD, dropout=TRANSFORMER_DROPOUT, num_summary_tokens=NUM_SUMMARY_TOKENS, activation="gelu"),
    pooler=CalibrationPoolerConfig(n_heads=POOLER_N_HEADS, dropout=POOLER_DROPOUT),
    mlp=MLPConfig(hidden_dims=MLP_HIDDEN_DIMS, output_dim=SHARED_FEATURE_DIM, activation="gelu", dropout=MLP_DROPOUT),
    head=PooledCalibrationHeadConfig(hidden_dims=HEAD_HIDDEN_DIMS, activation="gelu", dropout=HEAD_DROPOUT),
)

geometry_processor = GeometryProcessor()
tokenizer = Tokenizer(measurement_dim=MEASUREMENT_DIM, calibration_keys=CALIBRATION_KEYS)
model = PooledCalibrationModel(model_config).to(device=DEVICE, dtype=MODEL_DTYPE)
calibration_updater = CalibrationUpdater()
window_step = WindowStep(geometry_processor=geometry_processor, tokenizer=tokenizer, model=model, calibration_updater=calibration_updater)

if getattr(model, "uses_calibration_context", False):
    raise RuntimeError("PooledCalibrationModel must not receive numeric calibration context directly in its prediction head.")

print(model)
print()
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))

PooledCalibrationModel(
  (transformer_encoder): SummaryTokenTransformerEncoder(
    (input_projection): Linear(in_features=8, out_features=128, bias=True)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-11): 12 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (pooler): CalibrationPooler(
    (attention): MultiheadAttention(
      (out_proj): NonDynamicallyQ

## 6. Routing and pooled-output sanity check

In [8]:
sanity_base_window = next(iter(train_loader))
sanity_base_window = window_batch_to(sanity_base_window, DEVICE, dtype=MODEL_DTYPE)
sanity_generator = torch.Generator(device=DEVICE.type).manual_seed(TRAIN_AUGMENTATION_SEED)
sanity_window = augmentation_pipeline(sanity_base_window, generator=sanity_generator).augmented_window
sanity_result = window_step(sanity_window)

tokens = sanity_result.tokens
tokens.validate()

if tokens.calibration_ids is None:
    raise RuntimeError("Pooled calibration requires calibration_ids.")
if tuple(tokens.calibration_keys) != CALIBRATION_KEYS:
    raise RuntimeError(f"Tokenizer calibration order {tuple(tokens.calibration_keys)} does not match requested {CALIBRATION_KEYS}.")
if tuple(sanity_result.model_output.predictions) != CALIBRATION_KEYS:
    raise RuntimeError(f"Prediction key order {tuple(sanity_result.model_output.predictions)} does not match {CALIBRATION_KEYS}.")

print("Token shape:", tuple(tokens.x.shape))
print("Calibration keys:", tuple(tokens.calibration_keys))
print("Prediction keys:", tuple(sanity_result.model_output.predictions))
print("Pooled/shared feature shape:", None if sanity_result.model_output.shared_features is None else tuple(sanity_result.model_output.shared_features.shape))
print()

for calibration_id, calibration_key in enumerate(tokens.calibration_keys):
    counts = ((tokens.calibration_ids == calibration_id) & tokens.token_mask).sum(dim=1)
    prediction = sanity_result.model_output.predictions[calibration_key]
    print(f"{calibration_id}: {calibration_key}")
    print("  valid token count min/max:", int(counts.min()), int(counts.max()))
    print("  delta_xi:", tuple(prediction.delta_xi.shape))
    print("  delta_tau:", tuple(prediction.delta_tau.shape))

Token shape: (16, 1051, 8)
Calibration keys: ('imu', 'lidar')
Prediction keys: ('imu', 'lidar')
Pooled/shared feature shape: (16, 2, 128)

0: imu
  valid token count min/max: 1000 1002
  delta_xi: (16, 6)
  delta_tau: (16, 1)
1: lidar
  valid token count min/max: 48 49
  delta_xi: (16, 6)
  delta_tau: (16, 1)


## 7. Loss and optimizer

In [9]:
def correction_regression_loss(prediction, target):
    if CORRECTION_LOSS_TYPE == "mse":
        return F.mse_loss(prediction, target)
    if CORRECTION_LOSS_TYPE == "smooth_l1":
        return F.smooth_l1_loss(prediction, target, beta=1.0)
    if CORRECTION_LOSS_TYPE == "l1":
        return F.l1_loss(prediction, target)
    raise ValueError(f"Unknown CORRECTION_LOSS_TYPE={CORRECTION_LOSS_TYPE!r}.")


def compute_training_loss(window, model_output):
    if window.targets is None:
        raise ValueError("Training window has no targets.")

    total_loss = torch.zeros((), dtype=MODEL_DTYPE, device=DEVICE)
    metrics = {}
    num_calibrations = 0

    for calibration_key, prediction in model_output.predictions.items():
        if calibration_key not in window.targets:
            raise KeyError(f"Missing training target for calibration {calibration_key!r}.")

        target = window.targets[calibration_key]
        current_state = window.current_calibration[calibration_key]

        if target.next_transform is None or target.next_time_offset is None:
            raise ValueError(f"Calibration target {calibration_key!r} does not contain the final calibration state.")

        target_delta_transform = target.next_transform @ torch.linalg.inv(current_state.transform)
        target_delta_xi = se3_log(target_delta_transform)
        target_delta_tau = target.next_time_offset - current_state.time_offset

        rotation_loss = correction_regression_loss(prediction.delta_xi[:, :3] / ROTATION_LOSS_SCALE_RAD, target_delta_xi[:, :3] / ROTATION_LOSS_SCALE_RAD)
        translation_loss = correction_regression_loss(prediction.delta_xi[:, 3:] / TRANSLATION_LOSS_SCALE_M, target_delta_xi[:, 3:] / TRANSLATION_LOSS_SCALE_M)
        time_offset_loss = correction_regression_loss(prediction.delta_tau / TIME_OFFSET_LOSS_SCALE_S, target_delta_tau / TIME_OFFSET_LOSS_SCALE_S)

        if target.change_label is None:
            change_event_loss = torch.zeros((), dtype=MODEL_DTYPE, device=DEVICE)
            change_time_loss = torch.zeros((), dtype=MODEL_DTYPE, device=DEVICE)
            change_labels = None
        else:
            change_labels = target.change_label.to(dtype=prediction.change_event_logit.dtype)
            change_event_loss = F.binary_cross_entropy_with_logits(prediction.change_event_logit, change_labels)
            positive_event_mask = change_labels[:, 0] > 0.5
            if target.change_time is not None and torch.any(positive_event_mask):
                change_time_loss = F.smooth_l1_loss(prediction.change_time[positive_event_mask] / CHANGE_TIME_LOSS_SCALE_S, target.change_time[positive_event_mask] / CHANGE_TIME_LOSS_SCALE_S)
            else:
                change_time_loss = torch.zeros((), dtype=MODEL_DTYPE, device=DEVICE)

        calibration_loss = ROTATION_LOSS_WEIGHT * rotation_loss + TRANSLATION_LOSS_WEIGHT * translation_loss + TIME_OFFSET_LOSS_WEIGHT * time_offset_loss + CHANGE_EVENT_LOSS_WEIGHT * change_event_loss + CHANGE_TIME_LOSS_WEIGHT * change_time_loss
        total_loss = total_loss + calibration_loss
        num_calibrations += 1

        rotation_error_rad = torch.linalg.vector_norm(prediction.delta_xi[:, :3] - target_delta_xi[:, :3], dim=-1)
        translation_error_m = torch.linalg.vector_norm(prediction.delta_xi[:, 3:] - target_delta_xi[:, 3:], dim=-1)
        time_offset_error_s = torch.abs(prediction.delta_tau[:, 0] - target_delta_tau[:, 0])

        metrics[f"{calibration_key}/rotation_error_deg"] = torch.rad2deg(rotation_error_rad).mean().item()
        metrics[f"{calibration_key}/translation_error_cm"] = (100.0 * translation_error_m).mean().item()
        metrics[f"{calibration_key}/time_offset_error_ms"] = (1000.0 * time_offset_error_s).mean().item()
        metrics[f"{calibration_key}/loss_rotation"] = rotation_loss.detach().item()
        metrics[f"{calibration_key}/loss_translation"] = translation_loss.detach().item()
        metrics[f"{calibration_key}/loss_time_offset"] = time_offset_loss.detach().item()
        metrics[f"{calibration_key}/loss_change_event"] = change_event_loss.detach().item()
        metrics[f"{calibration_key}/loss_change_time"] = change_time_loss.detach().item()

        if change_labels is not None:
            predicted_labels = torch.sigmoid(prediction.change_event_logit[:, 0]) >= 0.5
            true_labels = change_labels[:, 0] >= 0.5
            metrics[f"{calibration_key}/event_accuracy"] = (predicted_labels == true_labels).float().mean().item()

    if num_calibrations == 0:
        raise ValueError("Model produced no calibration predictions.")

    total_loss = total_loss / num_calibrations
    metrics["loss_total"] = total_loss.detach().item()
    return total_loss, metrics


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=LR_REDUCTION_FACTOR, patience=LR_PATIENCE_EPOCHS)

## 8. Generator and metric helpers

In [10]:
def make_augmentation_generator(seed: int):
    return torch.Generator(device=DEVICE.type).manual_seed(seed)


def accumulate_metrics(metric_sums, metrics, batch_size):
    for key, value in metrics.items():
        metric_sums[key] = metric_sums.get(key, 0.0) + float(value) * batch_size


def normalize_metrics(metric_sums, num_samples):
    if num_samples <= 0:
        raise ValueError("num_samples must be positive.")
    return {key: value / num_samples for key, value in metric_sums.items()}

## 9. One-batch optimization smoke test

In [11]:
smoke_model_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
smoke_optimizer_state = optimizer.state_dict()

smoke_window = next(iter(train_loader))
smoke_window = window_batch_to(smoke_window, DEVICE, dtype=MODEL_DTYPE)
smoke_generator = make_augmentation_generator(TRAIN_AUGMENTATION_SEED + 1)
smoke_window = augmentation_pipeline(smoke_window, generator=smoke_generator).augmented_window

model.train()
optimizer.zero_grad(set_to_none=True)
smoke_result = window_step(smoke_window)
smoke_loss, smoke_metrics = compute_training_loss(smoke_window, smoke_result.model_output)
smoke_loss.backward()

gradient_norm = torch.sqrt(sum((parameter.grad.detach() ** 2).sum() for parameter in model.parameters() if parameter.grad is not None))
print("Smoke loss:", smoke_loss.item())
print("Gradient norm:", gradient_norm.item())
print("Prediction keys:", tuple(smoke_result.model_output.predictions))

if not torch.isfinite(smoke_loss):
    raise RuntimeError("Smoke loss is not finite.")
if not torch.isfinite(gradient_norm) or gradient_norm.item() <= 0.0:
    raise RuntimeError("Pooled model did not receive a finite nonzero gradient.")

model.load_state_dict(smoke_model_state)
optimizer.load_state_dict(smoke_optimizer_state)
optimizer.zero_grad(set_to_none=True)

Smoke loss: 94.3617935180664
Gradient norm: 713.4281005859375
Prediction keys: ('imu', 'lidar')


## 10. Epoch runner

In [12]:
def run_epoch(loader, *, training: bool, augmentation_generator: torch.Generator, global_step: int):
    model.train(training)
    metric_sums = {}
    num_samples = 0
    start_time = time.perf_counter()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    grad_context = torch.enable_grad() if training else torch.no_grad()

    with grad_context:
        for batch_index, base_window in enumerate(loader):
            base_window = window_batch_to(base_window, DEVICE, dtype=MODEL_DTYPE)
            training_window = augmentation_pipeline(base_window, generator=augmentation_generator).augmented_window

            if training:
                optimizer.zero_grad(set_to_none=True)

            step_result = window_step(training_window)
            loss, metrics = compute_training_loss(training_window, step_result.model_output)

            if training:
                loss.backward()
                if GRADIENT_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                optimizer.step()
                global_step += 1

            batch_size = next(iter(training_window.streams.values())).values.shape[0]
            accumulate_metrics(metric_sums, metrics, batch_size)
            num_samples += batch_size

            if training and PRINT_EVERY_BATCHES > 0 and (batch_index + 1) % PRINT_EVERY_BATCHES == 0:
                print(f"batch {batch_index + 1:5d}/{len(loader):5d} loss={metrics['loss_total']:.5f}")

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    elapsed_s = time.perf_counter() - start_time
    epoch_metrics = normalize_metrics(metric_sums, num_samples)
    epoch_metrics["epoch_time_s"] = elapsed_s
    epoch_metrics["samples_per_s"] = num_samples / elapsed_s
    return epoch_metrics, global_step

## 11. Checkpoint paths and experiment snapshot

In [13]:
RUN_DIR = ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_PATH = RUN_DIR / "best.pt"
LAST_CHECKPOINT_PATH = RUN_DIR / "last.pt"
METRICS_PATH = RUN_DIR / "metrics.jsonl"

EXPERIMENT_SNAPSHOT = {
    "run_name": RUN_NAME,
    "calibration_keys": CALIBRATION_KEYS,
    "correction_loss_type": CORRECTION_LOSS_TYPE,
    "rotation_loss_scale_rad": ROTATION_LOSS_SCALE_RAD,
    "translation_loss_scale_m": TRANSLATION_LOSS_SCALE_M,
    "time_offset_loss_scale_s": TIME_OFFSET_LOSS_SCALE_S,
    "architecture": "PooledCalibrationModel",
    "numeric_calibration_context_in_head": False,
    "paired_counterfactual_training": False,
    "observability_input": False,
    "observability_auxiliary_loss": False,
}

## 12. Optional resume

In [14]:
start_epoch = 0
global_step = 0
best_validation_loss = float("inf")

if RESUME_CHECKPOINT is not None:
    checkpoint = load_training_checkpoint(
        RESUME_CHECKPOINT,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        map_location=DEVICE,
        restore_rng_state=True,
        generators={"train_shuffle": train_shuffle_generator},
    )
    start_epoch = int(checkpoint.get("epoch", 0))
    global_step = int(checkpoint.get("global_step", 0))
    best_validation_loss = float(checkpoint.get("extra_state", {}).get("best_validation_loss", float("inf")))

    print("Resumed from:", RESUME_CHECKPOINT)
    print("start epoch:", start_epoch)
    print("global step:", global_step)

## 13. Main training loop

In [17]:
torch.cuda.empty_cache()

In [ ]:
history = []
train_augmentation_generator = make_augmentation_generator(TRAIN_AUGMENTATION_SEED)

with MetricsLogger(METRICS_PATH, append=RESUME_CHECKPOINT is not None) as logger:
    if RESUME_CHECKPOINT is None:
        logger.log_config(model_config, name="model_config")
        logger.log_config(augmentation_config, name="augmentation_config")
        logger.log_config(
            {
                "device": str(DEVICE),
                "batch_size": BATCH_SIZE,
                "window_duration_s": WINDOW_DURATION_S,
                "window_stride_s": WINDOW_STRIDE_S,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "train_sequences": TRAIN_NAMES,
                "validation_sequences": VAL_NAMES,
                "test_sequences": TEST_NAMES,
                **EXPERIMENT_SNAPSHOT,
            },
            name="experiment",
        )

    for epoch in range(start_epoch, NUM_EPOCHS):
        print()
        print("=" * 80)
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")

        train_metrics, global_step = run_epoch(train_loader, training=True, augmentation_generator=train_augmentation_generator, global_step=global_step)
        val_augmentation_generator = make_augmentation_generator(VALIDATION_AUGMENTATION_SEED)
        val_metrics, _ = run_epoch(val_loader, training=False, augmentation_generator=val_augmentation_generator, global_step=global_step)

        scheduler.step(val_metrics["loss_total"])
        learning_rate = optimizer.param_groups[0]["lr"]

        row = {
            "epoch": epoch + 1,
            "global_step": global_step,
            "learning_rate": learning_rate,
            **{f"train/{key}": value for key, value in train_metrics.items()},
            **{f"val/{key}": value for key, value in val_metrics.items()},
        }
        history.append(row)
        logger.log_metrics(
            {
                **train_metrics,
                "learning_rate": learning_rate,
            },
            split="train",
            epoch=epoch + 1,
            step=global_step,
        )

        logger.log_metrics(
            val_metrics,
            split="validation",
            epoch=epoch + 1,
            step=global_step,
        )

        print(f"train loss={train_metrics['loss_total']:.6f} val loss={val_metrics['loss_total']:.6f} lr={learning_rate:.3e}")
        # print(f"train time={train_metrics['epoch_time_s']:.1f}s val time={val_metrics['epoch_time_s']:.1f}s")

        if val_metrics["loss_total"] < best_validation_loss:
            best_validation_loss = val_metrics["loss_total"]
            save_training_checkpoint(
                BEST_CHECKPOINT_PATH,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch + 1,
                global_step=global_step,
                metrics=val_metrics,
                generators={"train_augmentation": train_augmentation_generator, "train_shuffle": train_shuffle_generator},
                extra_state={"best_validation_loss": best_validation_loss, "experiment": EXPERIMENT_SNAPSHOT},
            )
            print("Saved new best checkpoint.")

        if (epoch + 1) % CHECKPOINT_EVERY_EPOCHS == 0 or epoch + 1 == NUM_EPOCHS:
            save_training_checkpoint(
                LAST_CHECKPOINT_PATH,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch + 1,
                global_step=global_step,
                metrics=val_metrics,
                generators={"train_augmentation": train_augmentation_generator, "train_shuffle": train_shuffle_generator},
                extra_state={"best_validation_loss": best_validation_loss, "experiment": EXPERIMENT_SNAPSHOT},
            )
            print("Saved last checkpoint.")


Epoch 1/200
train loss=67.889834 val loss=66.145513 lr=3.000e-04
train time=100.4s val time=19.7s
Saved new best checkpoint.

Epoch 2/200


## 14. Training curves

In [ ]:
if history:
    epochs = [row["epoch"] for row in history]

    plt.figure(figsize=(12, 5))
    plt.plot(epochs, [row["train/loss_total"] for row in history], label="train")
    plt.plot(epochs, [row["val/loss_total"] for row in history], label="validation")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Pooled calibration training")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(epochs, [row["learning_rate"] for row in history])
    plt.xlabel("epoch")
    plt.ylabel("learning rate")
    plt.title("Learning-rate schedule")
    plt.grid(True)
    plt.show()

## 15. Evaluate the best checkpoint

In [ ]:
best_checkpoint = load_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, map_location=DEVICE, restore_rng_state=False)
test_augmentation_generator = make_augmentation_generator(TEST_AUGMENTATION_SEED)
test_metrics, _ = run_epoch(test_loader, training=False, augmentation_generator=test_augmentation_generator, global_step=global_step)

print("Best checkpoint epoch:", best_checkpoint.get("epoch"))
print("Test metrics:")
for key in sorted(test_metrics):
    print(f"  {key}: {test_metrics[key]:.6f}")

## 16. Selected-window correction diagnostics

In [ ]:
VIS_SEQUENCE_NAME = "Urban38"
VIS_START_WINDOW = 0
VIS_NUM_WINDOWS = 80
VIS_BATCH_SIZE = 16

sequence_indices = [index for index in range(len(test_dataset)) if test_dataset.sequence_name_for_window(index) == VIS_SEQUENCE_NAME]
if not sequence_indices:
    raise ValueError(f"Sequence {VIS_SEQUENCE_NAME!r} is not present in test_dataset.")

selected_indices = sequence_indices[VIS_START_WINDOW : VIS_START_WINDOW + VIS_NUM_WINDOWS]

print("Sequence:", VIS_SEQUENCE_NAME)
print("Selected windows:", len(selected_indices))
print("Dataset indices:", selected_indices[0], "...", selected_indices[-1])

In [ ]:
def collect_calibration_predictions(dataset, indices, *, batch_size, augmentation_seed):
    model.eval()
    augmentation_generator = make_augmentation_generator(augmentation_seed)
    collected = {
        calibration_key: {
            "reference_delta_xi": [],
            "predicted_delta_xi": [],
            "reference_delta_tau": [],
            "predicted_delta_tau": [],
        }
        for calibration_key in CALIBRATION_KEYS
    }

    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start : start + batch_size]
            base_window = collate_windows([dataset[index] for index in batch_indices])
            base_window = window_batch_to(base_window, DEVICE, dtype=MODEL_DTYPE)
            augmented_window = augmentation_pipeline(base_window, generator=augmentation_generator).augmented_window
            step_result = window_step(augmented_window)

            for calibration_key, prediction in step_result.model_output.predictions.items():
                target = augmented_window.targets[calibration_key]
                current_state = augmented_window.current_calibration[calibration_key]
                reference_delta_transform = target.next_transform @ torch.linalg.inv(current_state.transform)
                reference_delta_xi = se3_log(reference_delta_transform)
                reference_delta_tau = target.next_time_offset - current_state.time_offset

                collected[calibration_key]["reference_delta_xi"].append(reference_delta_xi.cpu())
                collected[calibration_key]["predicted_delta_xi"].append(prediction.delta_xi.cpu())
                collected[calibration_key]["reference_delta_tau"].append(reference_delta_tau.cpu())
                collected[calibration_key]["predicted_delta_tau"].append(prediction.delta_tau.cpu())

    for calibration_key in CALIBRATION_KEYS:
        for field_name in collected[calibration_key]:
            collected[calibration_key][field_name] = torch.cat(collected[calibration_key][field_name], dim=0).numpy()

    return collected


visualization_results = collect_calibration_predictions(test_dataset, selected_indices, batch_size=VIS_BATCH_SIZE, augmentation_seed=VISUALIZATION_AUGMENTATION_SEED)

In [ ]:
def print_initial_vs_model_errors(results, calibration_key):
    result = results[calibration_key]
    reference_xi = result["reference_delta_xi"]
    predicted_xi = result["predicted_delta_xi"]
    reference_tau = result["reference_delta_tau"][:, 0]
    predicted_tau = result["predicted_delta_tau"][:, 0]

    initial_rotation_error_deg = np.rad2deg(np.linalg.norm(reference_xi[:, :3], axis=1))
    model_rotation_error_deg = np.rad2deg(np.linalg.norm(predicted_xi[:, :3] - reference_xi[:, :3], axis=1))
    initial_translation_error_cm = 100.0 * np.linalg.norm(reference_xi[:, 3:], axis=1)
    model_translation_error_cm = 100.0 * np.linalg.norm(predicted_xi[:, 3:] - reference_xi[:, 3:], axis=1)
    initial_tau_error_ms = 1000.0 * np.abs(reference_tau)
    model_tau_error_ms = 1000.0 * np.abs(predicted_tau - reference_tau)

    print(calibration_key)
    print(f"rotation:    initial={initial_rotation_error_deg.mean():.3f} deg, model={model_rotation_error_deg.mean():.3f} deg")
    print(f"translation: initial={initial_translation_error_cm.mean():.3f} cm, model={model_translation_error_cm.mean():.3f} cm")
    print(f"time offset: initial={initial_tau_error_ms.mean():.3f} ms, model={model_tau_error_ms.mean():.3f} ms")


for calibration_key in CALIBRATION_KEYS:
    print_initial_vs_model_errors(visualization_results, calibration_key)
    print()

In [ ]:
def print_prediction_collapse_diagnostics(results, calibration_key):
    result = results[calibration_key]
    reference_xi = result["reference_delta_xi"]
    predicted_xi = result["predicted_delta_xi"]
    names = ("phi_x", "phi_y", "phi_z", "rho_x", "rho_y", "rho_z")

    print(calibration_key)
    print()

    for index, name in enumerate(names):
        target = reference_xi[:, index]
        prediction = predicted_xi[:, index]
        correlation = np.corrcoef(target, prediction)[0, 1] if np.std(target) > 0.0 and np.std(prediction) > 0.0 else np.nan
        print(f"{name:5s}  target std={np.std(target):.5f}  pred std={np.std(prediction):.5f}  corr={correlation:+.3f}")

    target_tau = result["reference_delta_tau"][:, 0]
    prediction_tau = result["predicted_delta_tau"][:, 0]
    correlation_tau = np.corrcoef(target_tau, prediction_tau)[0, 1] if np.std(target_tau) > 0.0 and np.std(prediction_tau) > 0.0 else np.nan
    print(f"{'tau':5s}  target std={np.std(target_tau):.5f}  pred std={np.std(prediction_tau):.5f}  corr={correlation_tau:+.3f}")


for calibration_key in CALIBRATION_KEYS:
    print_prediction_collapse_diagnostics(visualization_results, calibration_key)
    print()

## 17. Decisive measurement-dependence counterfactual

The numerical prior and original motion window are held fixed. One physical calibration is changed to `+20 deg` or `-20 deg`, and the corresponding measurements are re-expressed consistently. The desired corrections are opposite while the direct calibration context is identical.

A measurement-dependent model must distinguish the two cases.

In [ ]:
COUNTERFACTUAL_DATASET_INDEX = sequence_indices[0]
COUNTERFACTUAL_ROTATION_DEG = 20.0

counterfactual_base_window = window_batch_to(collate_windows([test_dataset[int(COUNTERFACTUAL_DATASET_INDEX)]]), DEVICE, dtype=MODEL_DTYPE)
counterfactual_prior = {calibration_key: CalibrationState(transform=state.transform.clone(), time_offset=state.time_offset.clone()) for calibration_key, state in counterfactual_base_window.current_calibration.items()}


def make_rotation_only_se3(rotation):
    transform = torch.zeros(rotation.shape[:-2] + (4, 4), dtype=rotation.dtype, device=rotation.device)
    transform[..., :3, :3] = rotation
    transform[..., 3, 3] = 1.0
    return transform


def reexpress_stream_for_true_rotation(stream, geometry_type, relative_rotation):
    values = stream.values.clone()

    if geometry_type == GeometryType.VECTOR:
        values = values @ relative_rotation
    elif geometry_type == GeometryType.SO3:
        values = relative_rotation.transpose(-1, -2)[:, None] @ values @ relative_rotation[:, None]
    elif geometry_type == GeometryType.SE3:
        relative_transform = make_rotation_only_se3(relative_rotation)
        inverse_relative_transform = torch.linalg.inv(relative_transform)
        values = inverse_relative_transform[:, None] @ values @ relative_transform[:, None]
    else:
        raise ValueError(f"Unsupported geometry type: {geometry_type!r}")

    return SensorStreamBatch(values=values, timestamps=stream.timestamps.clone(), sample_mask=stream.sample_mask.clone(), interval_start_timestamps=None if stream.interval_start_timestamps is None else stream.interval_start_timestamps.clone())


def make_counterfactual_true_rotation_window(base_window, calibration_key, delta_phi):
    old_state = base_window.current_calibration[calibration_key]
    delta_rotation = so3_exp(delta_phi)
    new_transform = old_state.transform.clone()
    new_transform[..., :3, :3] = delta_rotation @ old_state.transform[..., :3, :3]
    new_state = CalibrationState(transform=new_transform, time_offset=old_state.time_offset.clone())

    new_calibration = {key: CalibrationState(transform=state.transform.clone(), time_offset=state.time_offset.clone()) for key, state in base_window.current_calibration.items()}
    new_calibration[calibration_key] = new_state

    relative_rotation = old_state.transform[..., :3, :3].transpose(-1, -2) @ new_transform[..., :3, :3]
    new_streams = {}

    for stream_key, stream in base_window.streams.items():
        metadata = base_window.metadata[stream_key]

        if metadata.calibration_key == calibration_key:
            new_streams[stream_key] = reexpress_stream_for_true_rotation(stream, metadata.geometry_type, relative_rotation)
        else:
            new_streams[stream_key] = SensorStreamBatch(values=stream.values.clone(), timestamps=stream.timestamps.clone(), sample_mask=stream.sample_mask.clone(), interval_start_timestamps=None if stream.interval_start_timestamps is None else stream.interval_start_timestamps.clone())

    return WindowBatch(streams=new_streams, current_calibration=new_calibration, metadata=dict(base_window.metadata), targets=None, dataset_indices=base_window.dataset_indices), new_state


axis_names = ("x", "y", "z")
angle_rad = np.deg2rad(COUNTERFACTUAL_ROTATION_DEG)

model.eval()

print("=" * 120)
print("COUNTERFACTUAL TEST: SAME PRIOR, OPPOSITE TRUE CALIBRATION")
print("=" * 120)

with torch.no_grad():
    for calibration_key in CALIBRATION_KEYS:
        print(calibration_key)
        print("-" * 120)

        for axis_index, axis_name in enumerate(axis_names):
            delta_phi_positive = torch.zeros(1, 3, dtype=MODEL_DTYPE, device=DEVICE)
            delta_phi_negative = torch.zeros(1, 3, dtype=MODEL_DTYPE, device=DEVICE)
            delta_phi_positive[0, axis_index] = angle_rad
            delta_phi_negative[0, axis_index] = -angle_rad

            positive_window, positive_truth = make_counterfactual_true_rotation_window(counterfactual_base_window, calibration_key, delta_phi_positive)
            negative_window, negative_truth = make_counterfactual_true_rotation_window(counterfactual_base_window, calibration_key, delta_phi_negative)

            positive_result = window_step(positive_window, calibration=counterfactual_prior)
            negative_result = window_step(negative_window, calibration=counterfactual_prior)

            positive_prediction = positive_result.model_output.predictions[calibration_key].delta_xi[0, :3]
            negative_prediction = negative_result.model_output.predictions[calibration_key].delta_xi[0, :3]
            prior_state = counterfactual_prior[calibration_key]

            positive_target = se3_log(positive_truth.transform @ torch.linalg.inv(prior_state.transform))[0, :3]
            negative_target = se3_log(negative_truth.transform @ torch.linalg.inv(prior_state.transform))[0, :3]

            positive_prediction_deg = torch.rad2deg(positive_prediction).cpu().numpy()
            negative_prediction_deg = torch.rad2deg(negative_prediction).cpu().numpy()
            positive_target_deg = torch.rad2deg(positive_target).cpu().numpy()
            negative_target_deg = torch.rad2deg(negative_target).cpu().numpy()

            target_separation_deg = np.linalg.norm(positive_target_deg - negative_target_deg)
            prediction_separation_deg = np.linalg.norm(positive_prediction_deg - negative_prediction_deg)

            print(f"axis {axis_name}")
            print(f"  target + : {np.array2string(positive_target_deg, precision=3, suppress_small=True)} deg")
            print(f"  pred   + : {np.array2string(positive_prediction_deg, precision=3, suppress_small=True)} deg")
            print(f"  target - : {np.array2string(negative_target_deg, precision=3, suppress_small=True)} deg")
            print(f"  pred   - : {np.array2string(negative_prediction_deg, precision=3, suppress_small=True)} deg")
            print(f"  target +/- separation:    {target_separation_deg:.3f} deg")
            print(f"  predicted +/- separation: {prediction_separation_deg:.3f} deg")
            print()

        print()

## 18. Exact-truth fixed-point diagnostic

The current calibration supplied to `GeometryProcessor` is exactly the dataset truth. The correct predicted update is zero for every coordinate. This measures residual fixed-point bias that would accumulate during sequential rollout.

In [ ]:
FIXED_POINT_NUM_WINDOWS = min(100, len(sequence_indices))
fixed_point_positions = np.unique(np.linspace(0, len(sequence_indices) - 1, FIXED_POINT_NUM_WINDOWS).round().astype(int))
fixed_point_indices = np.asarray(sequence_indices)[fixed_point_positions]

fixed_point_results = {calibration_key: {"delta_xi": [], "delta_tau": []} for calibration_key in CALIBRATION_KEYS}

model.eval()

with torch.no_grad():
    for dataset_index in fixed_point_indices:
        window = window_batch_to(collate_windows([test_dataset[int(dataset_index)]]), DEVICE, dtype=MODEL_DTYPE)
        exact_true_calibration = {calibration_key: CalibrationState(transform=state.transform.clone(), time_offset=state.time_offset.clone()) for calibration_key, state in window.current_calibration.items()}
        result = window_step(window, calibration=exact_true_calibration)

        for calibration_key, prediction in result.model_output.predictions.items():
            fixed_point_results[calibration_key]["delta_xi"].append(prediction.delta_xi[0].detach().cpu())
            fixed_point_results[calibration_key]["delta_tau"].append(prediction.delta_tau[0, 0].detach().cpu())

for calibration_key in CALIBRATION_KEYS:
    fixed_point_results[calibration_key]["delta_xi"] = torch.stack(fixed_point_results[calibration_key]["delta_xi"]).numpy()
    fixed_point_results[calibration_key]["delta_tau"] = torch.stack(fixed_point_results[calibration_key]["delta_tau"]).numpy()

component_names = ("phi_x", "phi_y", "phi_z", "rho_x", "rho_y", "rho_z", "tau")

print("=" * 120)
print("EXACT-TRUTH FIXED-POINT DIAGNOSTIC")
print("=" * 120)

for calibration_key in CALIBRATION_KEYS:
    result = fixed_point_results[calibration_key]
    prediction = np.concatenate([result["delta_xi"], result["delta_tau"][:, None]], axis=1)

    print(calibration_key)
    print(f"{'component':8s} {'mean':>16s} {'mean |x|':>16s} {'RMSE':>16s} {'std':>16s} {'range':>16s}")

    for component_index, component_name in enumerate(component_names):
        values = prediction[:, component_index]

        if component_index < 3:
            values, unit = np.rad2deg(values), "deg"
        elif component_index < 6:
            values, unit = 100.0 * values, "cm"
        else:
            values, unit = 1000.0 * values, "ms"

        mean = np.mean(values)
        mean_abs = np.mean(np.abs(values))
        rmse = np.sqrt(np.mean(values ** 2))
        std = np.std(values)
        value_range = np.max(values) - np.min(values)

        print(f"{component_name:8s} {mean:+12.6f} {unit:3s} {mean_abs:12.6f} {unit:3s} {rmse:12.6f} {unit:3s} {std:12.6f} {unit:3s} {value_range:12.6f} {unit:3s}")

    print()

## 19. Interpretation gate

For this first pooled run, use these gates before adding more machinery:

1. **Held-out correction accuracy:** compare directly with the latest named-head MSE baseline.
2. **Counterfactual response:** the same prior with opposite physically consistent true rotations should produce clearly different predictions.
3. **Fixed point:** exact truth should produce corrections close to zero.
4. **Universality:** the model configuration contains no fixed calibration names or fixed number of prediction heads.

If these gates are healthy, the next clean observability experiment should be **oracle late fusion** of cached per-calibration observability into the pooled feature before the shared MLP/head. Compare that directly with this notebook before adding an auxiliary observability-prediction loss.

Paired `+delta/-delta` training and pair-difference loss should also remain a separate later experiment so their effect is not conflated with the architecture change.